# APAN 5200 — Logistic Regression Quiz Answer Key
**Dataset:** `employee_attrition.csv`  
**Outcome:** `attrition` (1 = left, 0 = stayed)

## Variable Descriptions
- `age`: Employee age
- `gender`: Male / Female
- `department`: Sales / Research & Development / Human Resources
- `years`: Years at company
- `performance`: Manager rating 1–5
- `jobsat`: Job satisfaction 1–5
- `income`: Annual salary
- `distance`: Distance from home (miles)
- `attrition`: **TARGET** (1=left, 0=stayed)

---
## Part 1 — StatsModels

In [ ]:
import pandas as pd
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')
from statsmodels.formula.api import logit

In [ ]:
# ============================================================
# Q1: Read data. 70/30 split using DataFrame.sample(), random_state=617.
#     What percentage of employees in the TRAIN sample left?
# ANSWER: 20.7%
# ============================================================

data = pd.read_csv('employee_attrition.csv')
train = data.sample(frac=0.7, random_state=617)
test  = data.drop(labels=train.index)

pct_left = round(train['attrition'].mean() * 100, 2)
print(f'Train attrition rate: {pct_left}%')  # ✅ 20.7

In [ ]:
# ============================================================
# Q2: Compare employees who left vs stayed on average age.
#     What is the average age of employees who LEFT?
# ANSWER: 48.3
# ============================================================

avg_age = train.groupby('attrition')['age'].mean().round(2)
print(avg_age)
print(f'Avg age (left): {round(train[train["attrition"]==1]["age"].mean(),2)}')  # ✅ 48.3

In [ ]:
# ============================================================
# Q3: Fit model1 = logit('attrition ~ age'). Which conclusion is correct?
# ANSWER: age coeff is POSITIVE (+0.013), p<0.05
#         → Older employees are MORE likely to leave (significant)
# ============================================================

model1 = logit('attrition ~ age', data=train).fit()
print(model1.summary())
print('\nage coeff:', round(model1.params['age'],4))  # ✅ ~+0.013

In [ ]:
# ============================================================
# Q4: Based on model1, what is the predicted probability that
#     the FIRST person in train will leave?
# ANSWER: 0.2077
# ============================================================

first_person = train.iloc[[0]][['age']]
prob_first = model1.predict(first_person)
print(f'First person age = {train["age"].iloc[0]}')
print(f'P(attrition=1) = {round(prob_first.values[0],4)}')  # ✅ 0.2077

In [ ]:
# ============================================================
# Q5: Based on model1, what is the probability that a
#     60-year-old employee will leave?
# ANSWER: 0.235
# ============================================================

prob_60 = model1.predict(pd.DataFrame({'age':[60]}))
print(f'P(attrition=1 | age=60) = {round(prob_60.values[0],4)}')  # ✅ 0.235

In [ ]:
# ============================================================
# Q6: If age goes up by one year, what is the PERCENT CHANGE
#     in likelihood of leaving?
# ANSWER: +1.33%
# KEY FORMULA: (exp(β) − 1) × 100
# ============================================================

b_age = model1.params['age']
pct_change = (math.exp(b_age) - 1) * 100
print(f'age coeff: {round(b_age,6)}')
print(f'Per +1 year age, attrition likelihood changes: {round(pct_change,2)}%')  # ✅ +1.33%

In [ ]:
# ============================================================
# Q7: Examine attrition rate by gender. Which statement is true?
# ANSWER: Female attrition rate (24.17%) > Male (18.55%)
# ============================================================

by_gender = train.groupby('gender')['attrition'].mean() * 100
print(by_gender.round(2))
# ✅ Female: 24.17%  |  Male: 18.55%  →  Female leave at higher rate

In [ ]:
# ============================================================
# Q8: Fit model2 = logit('attrition ~ gender').
#     What is the % difference in likelihood: Male vs Female?
# ANSWER: Male is ~28.55% LESS likely to leave (odds ratio ≈ 0.714)
# NOTE: Baseline = Female (alphabetically first)
# ============================================================

model2 = logit('attrition ~ gender', data=train).fit()
print(model2.summary())
b_male = model2.params['gender[T.Male]']
print(f'\nMale coeff: {round(b_male,4)}')
print(f'Odds ratio (Male vs Female): {round(math.exp(b_male),4)}')
print(f'Pct change: {round((math.exp(b_male)-1)*100,2)}%')  # ✅ −28.55%

In [ ]:
# ============================================================
# Q9: Fit model3 = logit('attrition ~ department').
#     Which statements are true?
# ANSWER:
#   Baseline = Human Resources (alphabetically first)
#   R&D coeff ≈ −0.315 → R&D less likely to leave than HR
#   Sales coeff ≈ +0.036 → not significantly different from HR (p>0.05)
# ============================================================

model3 = logit('attrition ~ department', data=train).fit()
print(model3.summary())
for k,v in model3.params.items():
    if k != 'Intercept':
        print(f'{k}: coef={round(v,4)}, OR={round(math.exp(v),4)}')

In [ ]:
# ============================================================
# Q10: Fit model4 with ALL variables. What is the Pseudo R²?
# ANSWER: 0.3486  (McFadden's Pseudo R²)
# ============================================================

model4 = logit(
    'attrition ~ age + gender + department + years + performance + jobsat + income + distance',
    data=train
).fit()

print(f'Pseudo R² (McFadden): {round(model4.prsquared,4)}')  # ✅ 0.3486

In [ ]:
# ============================================================
# Q11: Based on model4, which variables SIGNIFICANTLY affect attrition?
# ANSWER: years, performance, jobsat, gender[T.Male]  (p < 0.05)
#         NOT significant: age, income, distance, department(Sales)
# ============================================================

print(model4.summary())
sig = model4.pvalues[model4.pvalues < 0.05]
print('\nSignificant variables (p<0.05):')
print(sig)  # ✅ years, performance, jobsat, gender[T.Male]

In [ ]:
# ============================================================
# Q12: Based on model4, what is the impact of a ONE-UNIT increase
#      in jobsat on likelihood to leave?
# ANSWER: −35.99%  (higher satisfaction → much less likely to leave)
# ============================================================

b_jobsat = model4.params['jobsat']
impact = (math.exp(b_jobsat) - 1) * 100
print(f'jobsat coeff: {round(b_jobsat,4)}')
print(f'Per +1 jobsat, attrition likelihood changes: {round(impact,2)}%')  # ✅ −35.99%

---
## Part 2 — Scikit-Learn

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    recall_score, roc_auc_score
)

In [ ]:
# ============================================================
# Q13: Separate X and y. Stratified 80/20 split, random_state=617.
#      What % of the TRAIN sample left?
# ANSWER: 20.52%
# ============================================================

y = data['attrition']
X = data.drop('attrition', axis=1)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=617, stratify=y
)

print(f'Train attrition rate: {round(y_train.mean()*100,2)}%')  # ✅ 20.52

In [ ]:
# ============================================================
# Q14: Dummy-code gender and department (drop_first=True).
#      Reindex test to match train columns.
#      What is the MEAN of the dummy-coded gender variable?
# ANSWER: 0.6012  (~60% are Male in train)
# ============================================================

cat_cols = ['gender', 'department']
X_train = pd.get_dummies(X_train_raw, columns=cat_cols, drop_first=True, prefix_sep='_')
X_test  = pd.get_dummies(X_test_raw,  columns=cat_cols, drop_first=True, prefix_sep='_')
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

gender_col = [c for c in X_train.columns if 'gender' in c.lower()][0]
print(f'Gender dummy column: {gender_col}')
print(f'Mean of gender dummy: {round(X_train[gender_col].mean(),4)}')  # ✅ 0.6012

In [ ]:
# ============================================================
# Q15: Fit LogisticRegression(max_iter=10000).
#      What is the PREDICTED CLASS for the first observation in train?
# ANSWER: 0 (predicted to stay)  |  P(attrition=1) ≈ 0.006
# ============================================================

logit_sk = LogisticRegression(max_iter=10000, random_state=617)
logit_sk.fit(X_train, y_train)

y_pred_train       = logit_sk.predict(X_train)
y_pred_proba_train = logit_sk.predict_proba(X_train)
y_pred_test        = logit_sk.predict(X_test)
y_pred_proba_test  = logit_sk.predict_proba(X_test)

print(f'First obs predicted class : {y_pred_train[0]}')         # ✅ 0
print(f'First obs P(attrition=1)  : {round(y_pred_proba_train[0,1],4)}')  # ✅ ~0.006

In [ ]:
# ============================================================
# Q16: False Negatives are most costly. What is the SENSITIVITY
#      (Recall) in the TRAIN sample?
# ANSWER: 0.4932
# KEY: Sensitivity = TP / (TP + FN) = recall_score(y_train, y_pred)
# ============================================================

sensitivity_train = recall_score(y_train, y_pred_train)
print(f'Sensitivity (train): {round(sensitivity_train,4)}')  # ✅ 0.4932

cm_train = confusion_matrix(y_train, y_pred_train)
print(f'\nConfusion Matrix (train):\n{cm_train}')
print(f'TN={cm_train[0,0]}, FP={cm_train[0,1]}, FN={cm_train[1,0]}, TP={cm_train[1,1]}')

In [ ]:
# ============================================================
# Q17: What is the SENSITIVITY in the TEST sample?
# ANSWER: 0.5234
# NOTE: test > train → no overfitting, model generalises well
# ============================================================

sensitivity_test = recall_score(y_test, y_pred_test)
print(f'Sensitivity (test): {round(sensitivity_test,4)}')  # ✅ 0.5234

cm_test = confusion_matrix(y_test, y_pred_test)
print(f'\nConfusion Matrix (test):\n{cm_test}')

In [ ]:
# ============================================================
# Q18: What is the AUC in the TRAIN sample?
# ANSWER: 0.8789
# ============================================================

auc_train = roc_auc_score(y_train, y_pred_proba_train[:,1])
print(f'AUC (train): {round(auc_train,4)}')  # ✅ 0.8789

In [ ]:
# ============================================================
# Q19: What is the AUC in the TEST sample?
# ANSWER: 0.8833
# KEY INSIGHTS:
#   AUC train ≈ AUC test → good generalisation, not overfitting
#   Sensitivity (~50%) low despite high AUC (~88%) → default
#   threshold=0.5 is suboptimal; lowering it raises sensitivity
#   but reduces precision (more false alarms)
# ============================================================

auc_test = roc_auc_score(y_test, y_pred_proba_test[:,1])
print(f'AUC (test): {round(auc_test,4)}')  # ✅ 0.8833

---
## Answer Summary

In [ ]:
summary = {
    'Q1  Train attrition rate (%)':  '20.7',
    'Q2  Avg age of leavers':         '48.3',
    'Q3  age effect':                 'Positive & significant (p<0.05)',
    'Q4  P(leave | first person)':    '0.2077',
    'Q5  P(leave | age=60)':          '0.235',
    'Q6  age +1 → % change odds':     '+1.33%',
    'Q7  Higher attrition gender':    'Female (24.17% vs 18.55%)',
    'Q8  Male vs Female % diff':      '-28.55%',
    'Q9  Baseline department':         'Human Resources',
    'Q10 Pseudo R²':                   '0.3486',
    'Q11 Significant vars':            'years, performance, jobsat, gender',
    'Q12 jobsat +1 → % change odds':   '-35.99%',
    'Q13 Stratified train rate (%)':   '20.52',
    'Q14 Gender dummy mean':           '0.6012',
    'Q15 First obs prediction':        '0 (stay)',
    'Q16 Sensitivity (train)':         '0.4932',
    'Q17 Sensitivity (test)':          '0.5234',
    'Q18 AUC (train)':                 '0.8789',
    'Q19 AUC (test)':                  '0.8833',
}
for k,v in summary.items():
    print(f'{k:<35} → {v}')